In [24]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict,List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing_extensions import Annotated
import operator

In [25]:
load_dotenv()

True

In [26]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
) 

model = ChatHuggingFace(llm = llm)



 

In [27]:
class EssayState(TypedDict):
    topic: str
    essay: str

    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str

    individual_scores: Annotated[List[int], operator.add]
    avg_score :float


In [28]:
class EvaluationSchema(BaseModel):
    language_feedback: str = Field(description="Feedback on grammar and vocabulary")
    analysis_feedback: str = Field(description="Feedback on argument depth")
    clarity_feedback: str = Field(description="Feedback on structure and flow")
    overall_feedback: str = Field(description="Overall feedback for the essay")

    language_score: int = Field(ge=0, le=10)
    analysis_score: int = Field(ge=0, le=10)
    clarity_score: int = Field(ge=0, le=10)


In [30]:
structure_model = model.with_structured_output(EvaluationSchema)

NotImplementedError: Pydantic schema is not supported for function calling

In [ ]:
def generate_essay(state: EvaluationSchema):
    prompt = f"write a essay on the following topic :{state['topic']}"

    response = model.invoke(prompt)

    return {"essay" : response.content}

In [ ]:
def language(state:EvaluationSchema):
    prompt = f'evaluate the langauge quality of the foloowing essay and provide a feedback and assign a score out of 10\n of essay : {state['essay']}'

    